# 03 - Rule-based feature engineering (Person 2)

Builds the rule-based fraud signals from `src/features/rule_features.py` on the **train** split and checks
which of them actually separate fraud from real postings. Feature definitions: `docs/feature_descriptions.md`.

In [ ]:
# --- change these paths to match where your files are ---
REPO = "/content/p2"            # folder that contains src/
TRAIN = "/content/train.csv"

import sys
sys.path.insert(0, REPO)

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from src.features.rule_features import extract_features_df

df = pd.read_csv(TRAIN)
y = df["fraudulent"]
print(df.shape, "| fraud rate:", round(y.mean(), 4))

X = extract_features_df(df)
print("feature table:", X.shape)

## Which features separate fraud from real?
AUC of each feature used on its own as a score (0.5 = no signal; far above or below 0.5 = useful).

In [ ]:
rows = []
for c in X.columns:
    v = X[c]
    m = v.notna()
    auc = roc_auc_score(y[m], v[m]) if v[m].nunique() > 1 else np.nan
    rows.append({"feature": c, "mean_fraud": v[y == 1].mean(), "mean_real": v[y == 0].mean(),
                 "share_nonzero": (v.fillna(0) != 0).mean(), "auc": auc})
cmp = pd.DataFrame(rows)
cmp["signal_strength"] = (cmp["auc"] - 0.5).abs()
cmp = cmp.sort_values("signal_strength", ascending=False).round(3)
cmp.drop(columns="signal_strength").head(20)

In [ ]:
import matplotlib.pyplot as plt

top = cmp.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.barh(top["feature"], top["signal_strength"], color="#2a78d6", height=0.6)
ax.set_xlabel("|AUC - 0.5|  (0 = no signal)")
ax.set_title("Rule features that separate fraud from real (train)", loc="left", fontsize=10)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.grid(axis="x", color="#e4e3df", linewidth=0.6)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## How often does scam wording appear in this dataset?
The email-domain, URL and phone features cannot fire because the dataset masks them, and scam phrases may be rare.
This table shows the share of fraud vs real postings where each signal fires at all.

In [ ]:
scam_cols = ["payment_request_count", "messaging_app_count", "urgency_count", "too_good_count",
             "personal_info_count", "n_real_emails", "free_email_domain", "suspicious_url"]
pd.DataFrame({
    "fraud: % of postings with signal": (X.loc[y == 1, scam_cols] > 0).mean() * 100,
    "real: % of postings with signal": (X.loc[y == 0, scam_cols] > 0).mean() * 100,
}).round(2)

## Notes for the report
- Read the two tables above and write down: (1) which features are strongest, (2) which scam-wording features almost never fire.
- Missing-field features (company profile, logo) usually carry the signal in this dataset; scam wording is rare, and the masked
  contact details mean the email-domain features can only be used on pasted or uploaded postings in the app.
- Features were examined on the train split only, so the test set stays untouched.